# PS1: Real-Time Fake PTP Detection (Demo/Lightweight Pipeline)

This notebook demonstrates a lightweight pipeline to predict whether a Promise to Pay (PTP) will be kept or broken.
It uses simple tabular features and a basic classification model.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("Libraries imported successfully!")

## 2. Data Loading

In [ ]:
base_path = r"../../Dataset/data"
print(f"Loading data from: {base_path}")

In [ ]:
ptps = pd.read_csv(os.path.join(base_path, "ps1_fake_ptp", "ptps.csv"))
accounts = pd.read_csv(os.path.join(base_path, "shared", "accounts.csv"))
payments = pd.read_csv(os.path.join(base_path, "shared", "payments.csv"))

print(f"Loaded PTPs shape: {ptps.shape}")
print(f"Loaded Accounts shape: {accounts.shape}")
print(f"Loaded Payments shape: {payments.shape}")

In [ ]:
# Let's peek at the PTPs data
print("PTPs Data Sample:")
print(ptps.head())

In [ ]:
# Fix datetime formats
print("Converting timestamps to datetime objects...")
payments['payment_ts'] = pd.to_datetime(payments['payment_ts'])
ptps['captured_ts'] = pd.to_datetime(ptps['captured_ts'])
ptps['promised_date'] = pd.to_datetime(ptps['promised_date'])
print("Datetime conversion complete.")

## 3. Data Preprocessing & Feature Engineering

In [ ]:
print("Merging PTPs with Payments to find valid payments...")
ptp_payments = pd.merge(ptps, payments, on='account_id', how='left')
valid_payments = ptp_payments[
    (ptp_payments['payment_ts'] >= ptp_payments['captured_ts']) &
    (ptp_payments['payment_ts'] <= ptp_payments['promised_date'] + pd.Timedelta(days=3))
]
print(f"Found {len(valid_payments)} valid payments matching PTP windows.")

In [ ]:
# Sum up valid payments per PTP
payment_sums = valid_payments.groupby('ptp_id')['amount'].sum().reset_index().rename(columns={'amount': 'total_paid'})
ptps = pd.merge(ptps, payment_sums, on='ptp_id', how='left').fillna({'total_paid': 0})

# Create the target variable
print("Creating target variable 'kept'...")
ptps['kept'] = (ptps['total_paid'] >= (ptps['promised_amount'] * 0.9)).astype(int)
print(f"Target variable distribution:\n{ptps['kept'].value_counts(normalize=True)}")

In [ ]:
print("Merging with account data...")
data = pd.merge(ptps, accounts, on='account_id', how='left')
print(f"Final merged dataset shape: {data.shape}")

In [ ]:
# Select basic features
features = ['promised_amount', 'overdue_at_capture', 'emi_amount', 'outstanding', 
            'ability_to_pay_estimate', 'prev_ptp_count', 'prev_ptp_broken']

print("Cleaning numeric features...")
for col in features:
    data[col] = pd.to_numeric(data[col], errors='coerce').fillna(0)
    
print("Checking for nulls in features:")
print(data[features].isnull().sum())

In [ ]:
# Setup X and y
X = data[features]
y = data['kept']
print(f"Features shape: {X.shape}, Target shape: {y.shape}")

## 4. Model Training and Evaluation

In [ ]:
print("Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
print("Training RandomForestClassifier...")
clf = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=5)
clf.fit(X_train, y_train)
print("Training complete!")

In [ ]:
print("Generating predictions...")
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

In [ ]:
print("\n--- Model Evaluation ---")
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("Demo pipeline executed successfully.")